# 交易成本扩展：数据适用性、静态成本与策略敏感性

## 主要发现
十个选定交易日共306,483条有效双边证券分钟盘口；10万元买卖成本中位数9.27/8.87基点，100万元完整覆盖率62.12%/67.68%。共同样本的早期盘口分钟较齐全，但沪市委托缺失、近期时间与正业务序号回退阻止了精确事件回放。

策略使用同一日频执行模拟器完成18组约束情景，另有12组固定成交费用替换。三种资金规模均不允许现金为负；固定成交反事实的资金不足单独保留。结果不证明净超额收益或真实容量。

## 方法与假设
证券代码模37的确定性子样本，连续竞价每证券分钟先取最早记录，再检查双边、有序盘口。早期时钟相对09:30且保留午休；近期为HHMMSSmmm。1基点=0.01%，成本相对当时买卖中间价，含半价差和吃档，不含费用、排队、补单和自身冲击。

完整订单分位数仅包含十档足够覆盖的记录。名义规模=中间价×分析股数，允许非整手，不是现金预算。滞后特征用此前20个日频交易日，不证明历史发布时点可得。选定月份和共同存续证券不能代表全市场趋势。

In [ ]:
import os, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
run = Path(os.environ['EXECUTION_COST_EXPANSION_RUN'])
summary = json.loads((run/'l2-reviewed-v1/summary.json').read_text())
measurement_root = Path(summary['review']['measurement_source_root'])
output = run/'analysis-v1'
output.mkdir(exist_ok=True)
measurements = pd.concat([pd.read_parquet(p) for p in sorted(measurement_root.glob('measurements_*.parquet'))],ignore_index=True)
assert len(measurements)==summary['pooled']['snapshots']==306483
assert not measurements.duplicated(['TradingDay','SecuCode','minute']).any()
assert hashlib.sha256((run/'provenance/executed_expansion.py').read_bytes()).hexdigest()==summary['measurement_code_sha256']
assert all(str(r['history_end'])<str(r['day']) for r in summary['covariate_audit'])
pd.DataFrame(summary['audit'])[['day','sampled_snapshots','valid_books','one_sided','missing_security_minutes']]

## 从落盘测量独立复核
不调用研究汇总函数，直接重算成本分位数、覆盖率、相同样本与分组分母。源文件校验码和执行代码保留在私有产物中；本notebook源文件可以公开，执行版不进入Git。

In [ ]:
for row in summary['pooled']['cost_surface']:
    key=f"{row['side']}_{row['reference_notional_cny']}"
    cost=measurements[key+'_cost_bp'].dropna()
    coverage=measurements[key+'_coverage']
    assert len(cost)==row['full_order_cost_bp']['count']
    assert coverage.between(0,1).all()
    assert np.array_equal(measurements[key+'_cost_bp'].notna(),coverage.ge(1-1e-10))
    assert np.isclose(len(cost)/len(measurements),row['full_coverage_rate'])
    for name,q in [('p50',.5),('p90',.9),('p95',.95)]:
        assert np.isclose(cost.quantile(q),row['full_order_cost_bp'][name])
for group in ['by_adv_group','by_cap_group','by_volatility_group','by_time_bucket']:
    assert sum(v['snapshots'] for v in summary[group].values())==len(measurements)
cost_columns=[f'{side}_{size}_cost_bp' for side in ['buy','sell'] for size in [10000,100000,1000000,10000000]]
assert measurements[cost_columns].notna().all(axis=1).sum()==summary['pooled']['common_full_subset']['snapshots']
a,b=summary['comparison']['days']
pairs=measurements.loc[measurements.TradingDay.eq(a),['SecuCode','minute']].merge(measurements.loc[measurements.TradingDay.eq(b),['SecuCode','minute']],on=['SecuCode','minute'],validate='one_to_one')
assert len(pairs)==summary['comparison']['paired_valid_security_minutes']==23297
pd.DataFrame([{'side':r['side'],'reference_cny':r['reference_notional_cny'],'full_fraction':r['full_coverage_rate'],**r['full_order_cost_bp']} for r in summary['pooled']['cost_surface']])

## 数据适用性
106只共同证券的已观测分钟覆盖率为100%和96.99%，分母是106×237。2021沪市共同样本有成交但没有委托记录，深市候选编号匹配率99.68%/99.75%。早期没有通道/业务序号信息；近期正价格成交仍有时间倒退。订单编号是否存在不等于验证了订单语义或队列。

In [ ]:
event_rows=[]
for day,audit in summary['event_audit'].items():
    for market,group in audit['by_market'].items():
        reference=next(r for r in group['references'] if r['candidate']=='OrderID')
        event_rows.append({'day':day,'market':market,'order_rows':group['orders']['rows'],'trade_rows':group['trades']['rows'], 'buy_match':reference['buy']['match_rate'],'sell_match':reference['sell']['match_rate'], 'positive_trade_timestamp_regressions':group['positive_price_trades']['timestamp_regressions']})
pd.DataFrame(event_rows)

## 静态成本与覆盖率
下图是十个离散日期，不是连续时间趋势，也没有人口置信区间。不同金额的完整订单分位数会改变样本；第二幅限定到所有金额、买卖两侧均完整覆盖的20,709条盘口，仍是被选择的深盘口子样本。

In [ ]:
colors={'buy':'#326b86','sell':'#b06b38'}
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'axes.grid':False,'figure.facecolor':'white','axes.facecolor':'white','font.size':10})
def surface(value,side,size):
    return next(r for r in value['cost_surface'] if r['side']==side and r['reference_notional_cny']==size)
def save(fig,name,locale):
    fig.savefig(output/f'{name}.{locale}.svg',bbox_inches='tight')
    fig.savefig(output/f'{name}.{locale}.png',dpi=160,bbox_inches='tight')
    plt.close(fig)
for locale in ['en','zh']:
    zh=locale=='zh'
    with plt.rc_context({'font.sans-serif':['Noto Sans CJK SC','DejaVu Sans']}):
        days=sorted(summary['by_day']);x=np.arange(len(days));fig,axes=plt.subplots(2,1,figsize=(11,6.5),sharex=True,layout='constrained')
        for side,offset in [('buy',-.18),('sell',.18)]:
            label=('买入' if side=='buy' else '卖出') if zh else side.capitalize()
            axes[0].bar(x+offset,[surface(summary['by_day'][d],side,100000)['full_order_cost_bp']['p50'] for d in days],width=.34,color=colors[side],label=label)
            axes[1].bar(x+offset,[surface(summary['by_day'][d],side,1000000)['full_coverage_rate']*100 for d in days],width=.34,color=colors[side],label=label)
        axes[0].set_ylabel('10万元完整成本中位数（基点）' if zh else 'CNY 100k full-order median (bp)')
        axes[1].set_ylabel('100万元完整覆盖率（%）' if zh else 'CNY 1m full coverage (%)');axes[1].set_ylim(0,100)
        axes[1].set_xticks(x,labels=[f'{d[:4]}-{d[4:6]}-{d[6:]}' for d in days],rotation=35,ha='right')
        for ax in axes: ax.set_ylim(bottom=0);ax.grid(axis='y',alpha=.18);ax.legend(frameon=False,ncol=2)
        fig.suptitle('十个选定日期：成本须与覆盖率一起看' if zh else 'Ten selected dates: read conditional cost alongside coverage',fontweight='bold')
        save(fig,'expanded-daily',locale)
        fig,axes=plt.subplots(1,2,figsize=(10,4.4),layout='constrained');sizes=[10000,100000,1000000,10000000]
        for side in ['buy','sell']:
            label=('买入' if side=='buy' else '卖出') if zh else side.capitalize()
            axes[0].plot(sizes,[surface(summary['pooled']['common_full_subset'],side,q)['full_order_cost_bp']['p50'] for q in sizes],marker='o',color=colors[side],label=label)
            groups=['low','middle','high','unavailable'];xs=np.arange(4)+(-.17 if side=='buy' else .17)
            axes[1].bar(xs,[surface(summary['by_adv_group'][g],side,1000000)['full_coverage_rate']*100 for g in groups],width=.32,color=colors[side],label=label)
        axes[0].set_xscale('log');axes[0].set_xticks(sizes,labels=['1万','10万','100万','1000万'] if zh else ['10k','100k','1m','10m'])
        axes[0].set_xlabel('中间价名义金额（元）' if zh else 'Midpoint reference notional (CNY)');axes[0].set_ylabel('成本中位数（基点）' if zh else 'Median cost (bp)')
        axes[0].set_title('固定完整覆盖子样本：20,709条' if zh else 'Common fully covered subset: 20,709 books')
        axes[1].set_xticks(range(4),labels=['低','中','高','不可用'] if zh else ['Low','Middle','High','Unavailable']);axes[1].set_ylim(0,100)
        axes[1].set_ylabel('100万元完整覆盖率（%）' if zh else 'CNY 1m full coverage (%)');axes[1].set_title('此前20日平均成交额分组' if zh else 'Groups by prior 20-session ADV')
        for ax in axes: ax.set_ylim(bottom=0);ax.legend(frameon=False);ax.grid(axis='y',alpha=.18)
        save(fig,'expanded-size-groups',locale)
display(Image(filename=str(output/'expanded-daily.zh.png')))
display(Image(filename=str(output/'expanded-size-groups.zh.png')))
print('Rendered bilingual static-cost figures.')

## 现有策略：约束模拟与固定成交费用替换
56个交易日，日成交额5%参与率、T+1、涨跌停约束；碎股与复权收盘价代理。税费假设为每侧1基点综合佣金、卖出5基点印花税，最低佣金为零。原模拟5基点费用先移除，防止重复计费。参考成本包含后续2026年样本，只用于反事实敏感性；原输入发布时点和公司行动尚未核实。

累计未成交委托金额不是期末待成交量。改变费用会改变后续可用资金和订单，未成交总额下降不必然表示更容易成交。固定成交路径中的负现金仅表示资金缺口，不能当作可执行净值。

In [ ]:
baseline=run/'strategy/run-v1';expanded=run/'strategy/expanded-v1'
receipts=[json.loads((r/'receipt.json').read_text()) for r in [baseline,expanded]]
assert receipts[0]['original_reference_replay_max_nav_error_cny']==0
for root,receipt in zip([baseline,expanded],receipts):
    hashes=receipt.get('output_sha256',receipt.get('artifact_sha256',{}))
    for name,digest in hashes.items(): assert hashlib.sha256((root/name).read_bytes()).hexdigest()==digest
modeled=pd.concat([pd.read_csv(r/'modeled-summary.csv') for r in [baseline,expanded]],ignore_index=True)
assert len(modeled)==18 and modeled.negative_cash_sessions.eq(0).all()
for row in modeled.itertuples():
    root=expanded if row.scenario.startswith('fee_expanded') else baseline
    ledger=pd.read_csv(root/f'modeled-ledger-{int(row.capital_cny)}-{row.scenario}.csv')
    assert len(ledger)==56
    assert np.allclose(ledger.nav,ledger.cash+ledger.positions_value)
    assert np.isclose(ledger.nav.iloc[-1]/row.capital_cny,row.terminal_nav)
    assert (ledger.cash>=-1e-6).all()
overlays=pd.read_csv(baseline/'overlay-summary.csv')
assert overlays.fee_static_negative_cash_sessions.gt(0).all()
modeled[['capital_cny','scenario','terminal_nav','transaction_cost_cny','mean_one_way_turnover','unfilled_buy_cny']]

In [ ]:
scenarios=['gross','fee_only','fee_static_1m','fee_expanded_1m'];capitals=[10000000,100000000,1000000000]
palette=['#677b88','#859265','#bc9a56','#b16d45']
for locale in ['en','zh']:
    zh=locale=='zh'
    with plt.rc_context({'font.sans-serif':['Noto Sans CJK SC','DejaVu Sans']}):
        fig,axes=plt.subplots(1,2,figsize=(10.5,4.8),layout='constrained');x=np.arange(3)
        names=['无成本','假设税费','税费＋三日100万元参考','税费＋十日100万元参考'] if zh else ['Gross','Assumed fees','Fees + 3-day 1m costs','Fees + 10-day 1m costs']
        for i,scenario in enumerate(scenarios):
            rows=modeled.loc[modeled.scenario.eq(scenario)].set_index('capital_cny').loc[capitals]
            axes[0].bar(x+(i-1.5)*.19,(rows.terminal_nav-1)*100,width=.18,color=palette[i],label=names[i])
        gross=modeled.loc[modeled.scenario.eq('gross')].set_index('capital_cny').loc[capitals]
        completion=gross.executed_buy_cny/(gross.executed_buy_cny+gross.unfilled_buy_cny)*100
        axes[1].bar(x,completion,color='#326b86');axes[1].set_ylim(0,100)
        for i,value in enumerate(completion):axes[1].text(i,value+1,f'{value:.1f}%',ha='center')
        labels=['1000万元','1亿元','10亿元'] if zh else ['CNY 10m','CNY 100m','CNY 1bn']
        for ax in axes:ax.set_xticks(x,labels=labels);ax.set_ylim(bottom=0);ax.grid(axis='y',alpha=.18)
        axes[0].set_ylabel('样本累计模型回报（%）' if zh else 'Modeled cumulative sample return (%)');fig.legend(*axes[0].get_legend_handles_labels(),frameon=False,fontsize=9,loc='outside lower center',ncol=2)
        axes[1].set_ylabel('无成本模型累计买单完成比例（%）' if zh else 'Gross modeled cumulative buy completion (%)')
        fig.suptitle('56日诊断：资金与成本改变成交路径，未验证真实容量' if zh else '56-session diagnostic: capital and costs change fills; real capacity unverified',fontweight='bold',fontsize=11)
        save(fig,'strategy-capital',locale)
display(Image(filename=str(output/'strategy-capital.zh.png')))
stress=pd.read_csv(baseline/'turnover-stress.csv')
for row in stress.itertuples():
    assert row.sessions==250
    assert np.isclose(row.compounded_cost_fraction,1-(1-row.daily_cost_fraction)**250)
    assert np.isclose(row.linear_cost_fraction,row.daily_cost_fraction*250)
stress.loc[stress.capital_cny.eq(10000000),['anchor_notional_cny','linear_cost_fraction','compounded_cost_fraction']]

## 结论与未解决问题
早期深市是后续事件语义审计的优先候选；沪市不能补齐委托前直接回放。近期先调查时间与正序号回退，不通过重排原流掩盖问题。需要独立日期、供应商事件字典、PIT证券资格、真实委托回报，才能继续动态冲击和实际容量校准。

图表是描述性分布和模型情景，没有总体置信区间，不能证明策略有效性。